# ADNI SynthSeg mesh inspection

This notebook shows each requested SynthSeg structure in three separate forms:

1. **Raw**: direct marching-cubes mesh from the segmentation label.
2. **Minimal smooth**: the conservative, lightly smoothed correspondence input.
3. **Correspondence**: Deformetrica output with shared vertex order and faces.

No data are modified by this notebook. By default it shows both requested structures for up to three selected IDs; change the display controls below for a different subset.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import trimesh
from IPython.display import display
from plotly.subplots import make_subplots

OUTPUT_ROOT = Path('/home/jakaria/ADNI/ADNI_1_GO_Large/synthseg_minimal_correspondence/full')
CONFIG_PATH = OUTPUT_ROOT / "run_configuration.json"
MANIFEST_PATH = OUTPUT_ROOT / "manifests" / "selected_scans.csv"
if not CONFIG_PATH.is_file() or not MANIFEST_PATH.is_file():
    raise FileNotFoundError("Run configuration or selected manifest is missing.")

CONFIG = json.loads(CONFIG_PATH.read_text())
manifest = pd.read_csv(MANIFEST_PATH, dtype={"RID": str, "VISCODE": str, "scan_id": str})
structure_specs = CONFIG["structures"]
structure_names = [item["name"] for item in structure_specs]
print(f"Output root: {OUTPUT_ROOT}")
display(pd.DataFrame(structure_specs))
display(manifest[[column for column in ["scan_id", "RID", "VISCODE", "visit_dx_3class", "AGE", "PTGENDER"] if column in manifest.columns]])


Output root: /home/jakaria/ADNI/ADNI_1_GO_Large/synthseg_minimal_correspondence/full


,name,label,display_name,side
0,left_hippocampus,17,Left hippocampus,left
1,left_lateral_ventricle,4,Left lateral ventricle,left


,scan_id,RID,VISCODE,visit_dx_3class,AGE,PTGENDER
0,10_bl,10,bl,AD,73.900000,Female
1,10_m06,10,m06,AD,74.392813,Female
2,10_m12,10,m12,AD,74.896578,Female
3,10_m24,10,m24,AD,75.890418,Female
4,1001_bl,1001,bl,AD,69.300000,Male
...,...,...,...,...,...,...
4150,997_m36,997,m36,AD,83.428063,Female
4151,997_m48,997,m48,AD,84.416427,Female
4152,999_bl,999,bl,AD,70.800000,Male
4153,999_m06,999,m06,AD,71.528268,Male


## Output quality-control tables

In [2]:
validation_path = OUTPUT_ROOT / "reports" / "validation_summary.json"
if validation_path.is_file():
    validation = json.loads(validation_path.read_text())
    display(pd.DataFrame(validation.get("structures", {})).T)

qc_by_structure = {}
for structure in structure_names:
    path = OUTPUT_ROOT / structure / "mesh_qc.csv"
    qc_by_structure[structure] = pd.read_csv(path, dtype={"RID": str, "VISCODE": str, "scan_id": str}) if path.is_file() else pd.DataFrame()
    print(f"\n{structure}: {len(qc_by_structure[structure])} records")
    if not qc_by_structure[structure].empty:
        wanted = [
            "scan_id", "diagnosis", "status", "mask_volume_mm3", "raw_mesh_volume_mm3",
            "smooth_mesh_volume_mm3", "smooth_vs_mask_volume_pct", "raw_mesh_components",
            "smooth_mesh_components", "raw_watertight", "smooth_watertight",
            "smooth_watertight_strategy", "smooth_effective_closing_iterations",
            "smooth_effective_fill_holes", "smooth_watertight_attempts",
        ]
        display(qc_by_structure[structure][[column for column in wanted if column in qc_by_structure[structure].columns]])

lineage_path = OUTPUT_ROOT / "reports" / "mesh_volume_lineage.csv"
if lineage_path.is_file():
    lineage = pd.read_csv(lineage_path, dtype={"RID": str, "VISCODE": str, "scan_id": str})
    wanted = [
        "scan_id", "structure", "mask_volume_mm3", "raw_mesh_volume_mm3",
        "smooth_mesh_volume_mm3", "global_scale_factor", "rigid_linear_scale_factor",
        "correspondence_rescale_factor", "final_volume_mm3",
        "final_vs_smooth_volume_error_pct", "range_linear_scale_factor",
    ]
    print("\nVolume and coordinate lineage")
    display(lineage[[column for column in wanted if column in lineage.columns]])


,passed,failures,successful_meshes,failed_or_blocked_meshes,non_watertight_minimal_smooth_meshes,vertex_count,face_count,identical_face_connectivity,max_normalized_volume_error_pct,max_final_vs_smooth_volume_error_pct,scaled_obj_min,scaled_obj_max,scaled_obj_target_range
left_hippocampus,True,[],4155,0,[],2746,5488,True,0.000001,0.000002,-0.9,0.9,"[-0.9, 0.9]"
left_lateral_ventricle,True,[],4155,0,[],8346,16688,True,0.000001,0.000002,-0.9,0.9,"[-0.9, 0.9]"



left_hippocampus: 4155 records


,scan_id,diagnosis,status,mask_volume_mm3,raw_mesh_volume_mm3,smooth_mesh_volume_mm3,smooth_vs_mask_volume_pct,raw_mesh_components,smooth_mesh_components,raw_watertight,smooth_watertight,smooth_watertight_strategy,smooth_effective_closing_iterations,smooth_effective_fill_holes,smooth_watertight_attempts
0,10_bl,AD,ok,3395.999798,3362.249800,3321.016070,-2.208002,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
1,10_m06,AD,ok,3251.000000,3217.541667,3177.430052,-2.262994,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
2,10_m12,AD,ok,3262.000000,3228.416667,3184.029014,-2.390282,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
3,10_m24,AD,ok,3050.000000,3014.458333,2970.599234,-2.603304,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4,1001_bl,AD,ok,3073.000000,3040.458333,3003.660898,-2.256398,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4150,997_m36,AD,ok,2050.999878,2019.833213,1980.417065,-3.441386,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4151,997_m48,AD,ok,2110.000126,2080.083457,2041.119678,-3.264476,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4152,999_bl,AD,ok,2702.000000,2669.166667,2627.362744,-2.762297,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4153,999_m06,AD,ok,2721.999838,2691.833173,2650.121927,-2.640629,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing



left_lateral_ventricle: 4155 records


,scan_id,diagnosis,status,mask_volume_mm3,raw_mesh_volume_mm3,smooth_mesh_volume_mm3,smooth_vs_mask_volume_pct,raw_mesh_components,smooth_mesh_components,raw_watertight,smooth_watertight,smooth_watertight_strategy,smooth_effective_closing_iterations,smooth_effective_fill_holes,smooth_watertight_attempts
0,10_bl,AD,ok,15337.999086,15266.415757,14927.870484,-2.673938,2,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
1,10_m06,AD,ok,16672.000000,16600.000000,16270.929574,-2.405653,2,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
2,10_m12,AD,ok,17871.000000,17798.125000,17374.720090,-2.777013,2,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
3,10_m24,AD,ok,19296.000000,19225.041667,18828.707314,-2.421708,2,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4,1001_bl,AD,ok,23529.000000,23452.250000,23370.439665,-0.673893,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4150,997_m36,AD,ok,16289.999029,16226.832366,16164.232244,-0.772049,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4151,997_m48,AD,ok,17531.001045,17469.792708,17406.754936,-0.708722,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4152,999_bl,AD,ok,22851.000000,22786.291667,22717.730701,-0.583210,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing
4153,999_m06,AD,ok,26154.998441,26087.790112,26019.977198,-0.516235,1,1,True,True,requested_minimal_smoothing,0,False,requested_minimal_smoothing



Volume and coordinate lineage


,scan_id,structure,mask_volume_mm3,raw_mesh_volume_mm3,smooth_mesh_volume_mm3,global_scale_factor,rigid_linear_scale_factor,correspondence_rescale_factor,final_volume_mm3,final_vs_smooth_volume_error_pct,range_linear_scale_factor
0,10_bl,left_hippocampus,3395.999798,3362.249800,3321.016070,0.016699,1.0,0.999999,3321.016044,-7.642290e-07,2.438598
1,10_m06,left_hippocampus,3251.000000,3217.541667,3177.430052,0.016699,1.0,1.000789,3177.430042,-2.954527e-07,2.438598
2,10_m12,left_hippocampus,3262.000000,3228.416667,3184.029014,0.016699,1.0,0.999559,3184.029041,8.480531e-07,2.438598
3,10_m24,left_hippocampus,3050.000000,3014.458333,2970.599234,0.016699,1.0,0.998395,2970.599246,3.882977e-07,2.438598
4,1001_bl,left_hippocampus,3073.000000,3040.458333,3003.660898,0.016699,1.0,1.001055,3003.660915,5.488283e-07,2.438598
...,...,...,...,...,...,...,...,...,...,...,...
8305,997_m36,left_lateral_ventricle,16289.999029,16226.832366,16164.232244,0.006361,1.0,0.998300,16164.232266,1.350892e-07,2.295529
8306,997_m48,left_lateral_ventricle,17531.001045,17469.792708,17406.754936,0.006361,1.0,0.999384,17406.754627,-1.773291e-06,2.295529
8307,999_bl,left_lateral_ventricle,22851.000000,22786.291667,22717.730701,0.006361,1.0,1.003706,22717.730865,7.201133e-07,2.295529
8308,999_m06,left_lateral_ventricle,26154.998441,26087.790112,26019.977198,0.006361,1.0,1.004174,26019.977587,1.494102e-06,2.295529


## Inspect both structures for the selected pilot IDs

In [3]:
# The pilot displays every selected ID (at most three) and every requested structure.
# For a full run, change MAX_SCANS_TO_SHOW or SCAN_IDS deliberately before executing the next cell.
MAX_SCANS_TO_SHOW = 3
SCAN_IDS = manifest["scan_id"].astype(str).head(MAX_SCANS_TO_SHOW).tolist()
STRUCTURES_TO_SHOW = structure_names
PRIMARY_STRUCTURE = structure_names[0]  # Used by the correspondence overlay below.

if not SCAN_IDS:
    raise ValueError("The selected manifest has no scan IDs.")
if not set(STRUCTURES_TO_SHOW).issubset(structure_names):
    raise ValueError("STRUCTURES_TO_SHOW contains an unknown structure.")

def mesh_path(structure, scan_id, stage):
    base = OUTPUT_ROOT / structure
    if stage == "raw":
        return base / "raw_ply" / f"{scan_id}.ply"
    if stage == "minimal smooth":
        return base / "minimal_smooth_ply" / f"{scan_id}.ply"
    if stage == "correspondence mm":
        return base / "minimal_smooth_correspondence" / "final_ply_mm" / f"{scan_id}.ply"
    raise ValueError(stage)

for scan_id in SCAN_IDS:
    print(f"\n{scan_id}")
    for structure in STRUCTURES_TO_SHOW:
        for stage in ["raw", "minimal smooth", "correspondence mm"]:
            path = mesh_path(structure, scan_id, stage)
            print(f"  {structure:26s} | {stage:18s} | exists={path.is_file()}")



10_bl
  left_hippocampus           | raw                | exists=True
  left_hippocampus           | minimal smooth     | exists=True
  left_hippocampus           | correspondence mm  | exists=True
  left_lateral_ventricle     | raw                | exists=True
  left_lateral_ventricle     | minimal smooth     | exists=True
  left_lateral_ventricle     | correspondence mm  | exists=True

10_m06
  left_hippocampus           | raw                | exists=True
  left_hippocampus           | minimal smooth     | exists=True
  left_hippocampus           | correspondence mm  | exists=True
  left_lateral_ventricle     | raw                | exists=True
  left_lateral_ventricle     | minimal smooth     | exists=True
  left_lateral_ventricle     | correspondence mm  | exists=True

10_m12
  left_hippocampus           | raw                | exists=True
  left_hippocampus           | minimal smooth     | exists=True
  left_hippocampus           | correspondence mm  | exists=True
  left_lateral_ve

In [4]:
def stats(mesh):
    return {
        "vertices": len(mesh.vertices), "faces": len(mesh.faces),
        "components": len(mesh.split(only_watertight=False)), "watertight": bool(mesh.is_watertight),
        "euler_number": int(mesh.euler_number), "surface_area_mm2": float(mesh.area),
        "mesh_volume_mm3": abs(float(mesh.volume)),
    }

def add_mesh(figure, mesh, name, color, row, column, opacity=1.0, index_colors=False):
    vertices, faces = np.asarray(mesh.vertices), np.asarray(mesh.faces)
    options = dict(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], name=name, opacity=opacity,
        flatshading=False, showscale=False,
        lighting=dict(ambient=0.42, diffuse=0.75, specular=0.16, roughness=0.78),
    )
    if index_colors:
        options.update(intensity=np.arange(len(vertices)), colorscale="Turbo")
    else:
        options.update(color=color)
    figure.add_trace(go.Mesh3d(**options), row=row, col=column)

stages = ["raw", "minimal smooth", "correspondence mm"]
colors = {"raw": "#8E44AD", "minimal smooth": "#2E86DE", "correspondence mm": "#D35400"}
for scan_id in SCAN_IDS:
    rows = []
    figure = make_subplots(
        rows=len(STRUCTURES_TO_SHOW), cols=3,
        specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}] for _ in STRUCTURES_TO_SHOW],
        subplot_titles=[title for _ in STRUCTURES_TO_SHOW for title in ("Raw", "Minimal smooth", "Correspondence (mm)")],
        row_titles=[item["display_name"] for item in structure_specs if item["name"] in STRUCTURES_TO_SHOW],
    )
    for row_index, structure in enumerate(STRUCTURES_TO_SHOW, start=1):
        for column, stage in enumerate(stages, start=1):
            path = mesh_path(structure, scan_id, stage)
            if not path.is_file():
                continue
            mesh = trimesh.load(path, force="mesh", process=False)
            rows.append({"scan_id": scan_id, "structure": structure, "stage": stage, **stats(mesh)})
            add_mesh(figure, mesh, f"{structure} | {stage}", colors[stage], row_index, column)
    display(pd.DataFrame(rows))
    for index in range(1, len(STRUCTURES_TO_SHOW) * 3 + 1):
        scene = "scene" if index == 1 else f"scene{index}"
        figure.update_layout(**{scene: dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z")})
    figure.update_layout(
        title=f"Raw, minimal-smooth, and correspondence meshes | {scan_id}",
        width=1550, height=480 * len(STRUCTURES_TO_SHOW), template="plotly_white",
        margin=dict(l=0, r=0, t=65, b=0),
    )
    figure.show()


,scan_id,structure,stage,vertices,faces,components,watertight,euler_number,surface_area_mm2,mesh_volume_mm3
0,10_bl,left_hippocampus,raw,2703,5422,1,True,-8,1980.955975,3362.249723
1,10_bl,left_hippocampus,minimal smooth,2698,5392,1,True,2,1846.401091,3321.016026
2,10_bl,left_hippocampus,correspondence mm,2746,5488,1,True,2,1810.518307,3321.016034
3,10_bl,left_lateral_ventricle,raw,7942,15876,2,True,4,5700.763408,15266.415084
4,10_bl,left_lateral_ventricle,minimal smooth,7562,15120,1,True,2,5182.451362,14927.870028
5,10_bl,left_lateral_ventricle,correspondence mm,8346,16688,1,True,2,5259.475884,14927.869923


,scan_id,structure,stage,vertices,faces,components,watertight,euler_number,surface_area_mm2,mesh_volume_mm3
0,10_m06,left_hippocampus,raw,2721,5458,1,True,-8,1953.484019,3217.541667
1,10_m06,left_hippocampus,minimal smooth,2710,5416,1,True,2,1813.360509,3177.430039
2,10_m06,left_hippocampus,correspondence mm,2746,5488,1,True,2,1782.764344,3177.430060
3,10_m06,left_lateral_ventricle,raw,8114,16220,2,True,4,5898.489562,16600.000000
4,10_m06,left_lateral_ventricle,minimal smooth,7736,15468,1,True,2,5386.037293,16270.929619
5,10_m06,left_lateral_ventricle,correspondence mm,8346,16688,1,True,2,5450.718993,16270.929660


,scan_id,structure,stage,vertices,faces,components,watertight,euler_number,surface_area_mm2,mesh_volume_mm3
0,10_m12,left_hippocampus,raw,2722,5464,1,True,-10,1962.434473,3228.416667
1,10_m12,left_hippocampus,minimal smooth,2710,5416,1,True,2,1820.475023,3184.029021
2,10_m12,left_hippocampus,correspondence mm,2746,5488,1,True,2,1786.418289,3184.029040
3,10_m12,left_lateral_ventricle,raw,8532,17056,2,True,4,6181.794736,17798.125000
4,10_m12,left_lateral_ventricle,minimal smooth,8068,16132,1,True,2,5588.778777,17374.720137
5,10_m12,left_lateral_ventricle,correspondence mm,8346,16688,1,True,2,5625.737184,17374.720211


## Correspondence overlay

In [5]:
# Full runs may contain thousands of meshes. Keep this small for interactive viewing.
MAX_SCANS = 3
overlay_rows = manifest[["scan_id", "visit_dx_3class"]].head(MAX_SCANS)
diagnosis_colors = {"CN": "#2E86DE", "AD": "#C0392B", "MCI": "#F39C12"}
overlay = go.Figure()
overlay_stats = []
for _, row in overlay_rows.iterrows():
    scan_id = str(row["scan_id"])
    path = OUTPUT_ROOT / PRIMARY_STRUCTURE / "minimal_smooth_correspondence" / "final_ply" / f"{scan_id}.ply"
    if not path.is_file():
        continue
    mesh = trimesh.load(path, force="mesh", process=False)
    vertices, faces = np.asarray(mesh.vertices), np.asarray(mesh.faces)
    diagnosis = str(row["visit_dx_3class"])
    overlay.add_trace(go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], name=f"{scan_id} | {diagnosis}",
        color=diagnosis_colors.get(diagnosis, "#6C757D"), opacity=0.45, flatshading=False, showscale=False,
    ))
    overlay_stats.append({"scan_id": scan_id, "diagnosis": diagnosis, **stats(mesh)})
display(pd.DataFrame(overlay_stats))
overlay.update_layout(
    title=f"Correspondence overlay: {PRIMARY_STRUCTURE}", width=900, height=700, template="plotly_white",
    scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"), margin=dict(l=0, r=0, t=55, b=0),
)
overlay.show()


,scan_id,diagnosis,vertices,faces,components,watertight,euler_number,surface_area_mm2,mesh_volume_mm3
0,10_bl,AD,2746,5488,1,True,2,0.504869,0.015464
1,10_m06,AD,2746,5488,1,True,2,0.497130,0.014796
2,10_m12,AD,2746,5488,1,True,2,0.498149,0.014827


## Direct point-correspondence check

In [6]:
final_paths = sorted((OUTPUT_ROOT / PRIMARY_STRUCTURE / "minimal_smooth_correspondence" / "final_ply").glob("*.ply"))
if len(final_paths) < 2:
    print("At least two correspondence meshes are needed for this check.")
else:
    reference = trimesh.load(final_paths[0], force="mesh", process=False)
    rows = []
    index_figure = go.Figure()
    for path in final_paths[:min(3, len(final_paths))]:
        mesh = trimesh.load(path, force="mesh", process=False)
        rows.append({
            "scan_id": path.stem, "vertices": len(mesh.vertices), "faces": len(mesh.faces),
            "faces_identical_to_first": bool(np.array_equal(mesh.faces, reference.faces)),
        })
        vertices, faces = np.asarray(mesh.vertices), np.asarray(mesh.faces)
        index_figure.add_trace(go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], intensity=np.arange(len(vertices)),
            colorscale="Turbo", showscale=False, opacity=0.55, name=path.stem,
        ))
    display(pd.DataFrame(rows))
    index_figure.update_layout(
        title=f"Vertex-index colors: {PRIMARY_STRUCTURE}", width=900, height=700, template="plotly_white",
        scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=55, b=0),
    )
    index_figure.show()


,scan_id,vertices,faces,faces_identical_to_first
0,1001_bl,2746,5488,True
1,1001_m06,2746,5488,True
2,1001_m12,2746,5488,True
